# Decoder Locality Analysis

## Goal (H2: patch token-edit on decoder)

We edit token IDs inside a contiguous **token-space** patch (`patch_bbox_tok`) and decode to pixels.

The question is: does the decoded image change mostly inside the corresponding **pixel-space** patch (`patch_bbox_px`)?

For each image:

1. Encode the image with the VQGAN encoder to a discrete token grid
   $$
   z \in {0,\dots,K-1}^{H_{\text{tok}}\times W_{\text{tok}}}.
   $$

2. Choose a contiguous token patch $B_{\text{tok}}$ and create edited grids $\tilde z^{(m)}$ by replacing token IDs **only inside** that patch, for each token-edit mode $m \in \texttt{token_edit_modes}$.

3. Decode the clean and edited grids:

* $x_{\text{clean}} = D(z)$
* $x_{\text{edit}}^{(m)} = D(\tilde z^{(m)})$

4. Measure locality in **pixel space**: does the change $|x_{\text{edit}}^{(m)} - x_{\text{clean}}|$ stay mostly inside the projected pixel box `patch_bbox_px`?

## Record format

Each JSONL row corresponds to one ImageNet validation image (identified by `image_id`) and one H2 run that can include **multiple** token-edit modes.

* `token_edit_modes` (list of strings)
  Token-edit strategies applied to the patch (e.g. `"random_uniform"`, `"closest"`, `"farthest"`, `"orthogonal"`). For each mode $m$, the run produces an edited token grid and a decoded reconstruction.

* `token_grid_hw = [H_tok, W_tok]`
  Spatial resolution of the latent token grid (e.g. `[16, 16]`).

* `patch_bbox_tok = [j0, i0, j1, i1]`
  Token-space axis-aligned rectangle (aligned to the token grid).
  Convention: `i` is row, `j` is column. Edited token positions are:
  $$
  {(i,j);|; i_0 \le i < i_1,; j_0 \le j < j_1}.
  $$

* `patch_bbox_px = [x0, y0, x1, y1]`
  Pixel-space rectangle corresponding to `patch_bbox_tok`, projected into decoded image coordinates. Since the patch is token-aligned, this should match the token-box projection (up to rounding).

* `indices_clean` (length $H_{\text{tok}}W_{\text{tok}}$)
  Flattened clean token grid (row-major).

* `indices_edit_by_mode` (dict: mode → flattened grid)
  Mapping from each edit mode $m$ to the flattened edited token grid (same length as `indices_clean`).

> Backward-compat: if only one mode is present, some runs may also include `token_edit_mode` and `indices_edit`.

* Images on disk (under `images/<class>/<image_id>/`):

  * `0_original.png`: original ImageNet image (reference)
  * `1_recon_clean.png`: VQGAN reconstruction from `indices_clean`
  * `2_recon_token_edit_<mode>.png`: VQGAN reconstruction for each mode $m$ from `indices_edit_by_mode[m]`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple

import json
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from PIL import Image

from tqdm.auto import tqdm

import random

In [ ]:
SEED = 42
random.seed(SEED)

In [ ]:
def load_img_float01(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0


## Sanity Check

In [ ]:
# Add new models here later
EXPERIMENT_SPECS = {
    "vqgan":    Path("/exp/1767746258_robustness_dataset_vqgan_h2_patch_token_edit_decoder_patch8_seed0"),
    "llamagen": Path("/exp/1767394121_robustness_dataset_llamagen_h2_patch_token_edit_decoder_patch8_seed0"),
}

MAX_RECORDS = None  # optional global cap for debugging (applies per experiment when building the small index)

@dataclass
class Experiment:
    name: str
    outdir: Path
    images_dir: Path
    meta_files: list

def build_experiment(name: str, outdir: Path) -> Experiment:
    images_dir = outdir / "images"
    meta_files = sorted(outdir.glob("metadata_part_*.jsonl"))

    print(f"\n[{name}]")
    print("OUTDIR:", outdir)
    print("IMAGES_DIR exists:", images_dir.exists())
    print("Found metadata files:", len(meta_files))
    for p in meta_files:
        print(" -", p.name)

    assert outdir.exists(), f"[{name}] OUTDIR does not exist"
    assert images_dir.exists(), f"[{name}] Missing folder: {images_dir}"
    assert len(meta_files) > 0, f"[{name}] No metadata_part_*.jsonl files found"

    return Experiment(name=name, outdir=outdir, images_dir=images_dir, meta_files=meta_files)

EXPERIMENTS = {name: build_experiment(name, outdir) for name, outdir in EXPERIMENT_SPECS.items()}

In [ ]:
# ---- utilities ----

def _normalize_modes_and_indices(rec):
    modes = rec.get("token_edit_modes", None)
    indices_by_mode = rec.get("indices_edit_by_mode", None)

    if modes is None or indices_by_mode is None:
        # legacy fallback
        m = rec.get("token_edit_mode", None)
        z = rec.get("indices_edit", None)
        if m is not None and z is not None:
            modes = [m]
            indices_by_mode = {m: z}
        else:
            modes = [] if modes is None else list(modes)
            indices_by_mode = {} if indices_by_mode is None else dict(indices_by_mode)

    return list(modes), indices_by_mode

def iter_meta_records(meta_files, *, keep_indices: bool = False, max_records: int | None = None):
    """Stream records across all meta files. Safe for large-scale runs."""
    n = 0
    for fp in meta_files:
        with open(fp, "r") as f:
            for line in f:
                if not line.strip():
                    continue
                rec = json.loads(line)

                image_id = rec.get("image_id", None)
                bb_px  = rec.get("patch_bbox_px", None)
                bb_tok = rec.get("patch_bbox_tok", None)
                hw_tok = rec.get("token_grid_hw", None)
                modes, indices_by_mode = _normalize_modes_and_indices(rec)

                row = {
                    "image_id": image_id,
                    "patch_bbox_px": bb_px,
                    "patch_bbox_tok": bb_tok,
                    "token_grid_hw": hw_tok,
                    "token_edit_modes": modes,
                }
                if keep_indices:
                    row["indices_clean"] = rec.get("indices_clean", None)
                    row["indices_edit_by_mode"] = indices_by_mode

                yield row
                n += 1
                if max_records is not None and n >= max_records:
                    return

def _parse_one_meta_file_minimal(fp_str: str):
    """Parallel helper: minimal rows only (no indices)."""
    rows = []
    with open(fp_str, "r") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            modes, _ = _normalize_modes_and_indices(rec)
            rows.append({
                "image_id": rec.get("image_id", None),
                "patch_bbox_px": rec.get("patch_bbox_px", None),
                "patch_bbox_tok": rec.get("patch_bbox_tok", None),
                "token_grid_hw": rec.get("token_grid_hw", None),
                "token_edit_modes": modes,
            })
    return rows

def build_index(meta_files, *, max_records: int | None = None):
    """
    Build a small in-memory index for convenience (id + bbox + modes).
    Keeps RAM low so multiple experiments fit.
    """
    if max_records is not None:
        # sequential to enforce a true global cap
        rows = list(iter_meta_records(meta_files, keep_indices=True, max_records=max_records))
    else:
        workers = min(len(meta_files), os.cpu_count() or 4)
        rows = []
        with ProcessPoolExecutor(max_workers=workers) as ex:
            futs = [ex.submit(_parse_one_meta_file_minimal, str(fp)) for fp in meta_files]
            for fut in as_completed(futs):
                rows.extend(fut.result())

    rows.sort(key=lambda r: (r["image_id"] or ""))
    return rows

In [ ]:
records_index = {}
bbox_map = {}

for name, exp in EXPERIMENTS.items():
    idx = build_index(exp.meta_files, max_records=MAX_RECORDS)
    records_index[name] = idx
    bbox_map[name] = {r["image_id"]: r["patch_bbox_px"] for r in idx if r["image_id"] and r["patch_bbox_px"]}

    print(f"\n[{name}] index rows:", len(idx))
    if idx:
        print("Example keys:", list(idx[0].keys()))
        print("Example modes:", idx[0].get("token_edit_modes"))
        print("Example image_id:", idx[0].get("image_id"))

In [ ]:
# Δ heatmap sanity — all experiments x all modes (robust to Experiment objects)

BBoxPx = Tuple[int, int, int, int]

def delta_map_l1_mean(x_clean: np.ndarray, x_edit: np.ndarray) -> np.ndarray:
    # channel-averaged L1
    return np.mean(np.abs(x_edit - x_clean), axis=-1)

def delta_map_rmse(x_clean: np.ndarray, x_edit: np.ndarray) -> np.ndarray:
    diff = x_edit - x_clean
    return np.sqrt(np.mean(diff * diff, axis=-1))  # per-pixel RMSE across channels


def _add_bbox_rect(ax, bbox_px: Optional[BBoxPx], *, edgecolor="cyan", lw=2) -> None:
    if bbox_px is None:
        return
    x0, y0, x1, y1 = map(int, bbox_px)
    rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=lw, fill=False, edgecolor=edgecolor)
    ax.add_patch(rect)

def _exp_get(exp: Any, key: str, default=None):
    if isinstance(exp, dict):
        return exp.get(key, default)
    return getattr(exp, key, default)

def _iter_experiments(experiments: Any):
    """
    Accepts dict {name: exp} or list [exp,...].
    Prefers global EXPERIMENTS/experiments if None.
    """
    if experiments is None:
        experiments = globals().get("EXPERIMENTS", None) or globals().get("experiments", None)
    if isinstance(experiments, dict):
        for name, exp in experiments.items():
            yield str(name), exp
        return
    for i, exp in enumerate(experiments or []):
        name = _exp_get(exp, "name", None) or _exp_get(exp, "experiment_name", None) or f"exp_{i:02d}"
        yield str(name), exp

def _resolve_images_root(exp_cfg: Any) -> Path:
    """
    Supports:
      - Path / str directly
      - dict-like configs
      - Experiment objects (attributes)
    Tries common fields first; falls back to outdir/images.
    """
    if isinstance(exp_cfg, Path):
        return exp_cfg
    if isinstance(exp_cfg, str):
        return Path(exp_cfg)

    # helper to read either dict keys or object attrs
    def pick(obj: Any, k: str):
        if isinstance(obj, dict):
            return obj.get(k, None)
        return getattr(obj, k, None)

    # fields that may already point at the images directory
    for k in ("images_root", "IMAGES_DIR", "images_dir", "image_dir", "images"):
        v = pick(exp_cfg, k)
        if v is not None:
            p = Path(v)
            # accept even if it doesn't exist yet; existence checked later per-file
            return p

    # fall back: outdir -> outdir/images (your on-disk layout)
    for k in ("outdir", "OUTDIR", "output_dir", "run_dir", "root"):
        v = pick(exp_cfg, k)
        if v is not None:
            out = Path(v)
            cand = out / "images"
            return cand if cand.exists() else out

    raise ValueError(f"Unsupported experiment config for images_root: {type(exp_cfg)}")

def delta_maps_for_one_experiment(
    *, image_id: str, modes: Sequence[str], images_root: Path
) -> List[Tuple[str, Optional[np.ndarray]]]:
    """
    Returns [(mode, delta_map_or_None), ...] in the same order as `modes`.
    Requires: paths_for_sample(images_root, image_id, modes=...) and load_img_float01(path)
    """
    ps = paths_for_sample(images_root, image_id, modes=modes)
    if not ps["recon_clean"].exists():
        return [(m, None) for m in modes]

    x_clean = load_img_float01(ps["recon_clean"])
    out: List[Tuple[str, Optional[np.ndarray]]] = []

    mode_to_path = ps["recon_edits"]
    for m in modes:
        p = mode_to_path.get(m, None)
        if p is None or (not p.exists()):
            out.append((m, None))
            continue
        x_edit = load_img_float01(p)
        out.append((m, delta_map_rmse(x_clean, x_edit)))

    return out

In [ ]:
# ---- prep for cross-experiment visual comparison ----

# keep the same setting if it already exists; otherwise default to metadata-defined modes
AUTO_DETECT_MODES_FROM_DISK = globals().get("AUTO_DETECT_MODES_FROM_DISK", False)

# stable experiment order
exp_names = list(EXPERIMENTS.keys())

# experiment -> {image_id: row}
row_by_exp = {
    exp_name: {
        r["image_id"]: r
        for r in records_index[exp_name]
        if r.get("image_id") is not None
    }
    for exp_name in exp_names
}

# image ids present in all experiments
shared_ids = sorted(
    set.intersection(*[set(row_by_exp[name].keys()) for name in exp_names])
)

print("exp_names:", exp_names)
print("n_shared_ids:", len(shared_ids))
print("sample shared_ids:", shared_ids[:5])

In [ ]:
import random
rng = random.Random(SEED)

# ---- config
K_IMAGES = 5
HEADER_H = 0.35
ROW_H = 3.8
COL_W = 4.8
HEATMAP_CMAP = "magma"

# ---- sample ids
pick = rng.sample(shared_ids, k=min(K_IMAGES, len(shared_ids)))
print("picked:", len(pick), pick[:3])

def _span_split(ncols: int):
    left = max(1, ncols // 2)
    right = max(1, ncols - left)
    return left, right

for j, image_id in enumerate(pick):
    # choose canonical row (for bbox + modes) from first exp that has this image_id
    canonical_row = None
    for name in exp_names:
        r = row_by_exp[name].get(image_id)
        if r is not None:
            canonical_row = r
            break

    bbox_px = canonical_row.get("patch_bbox_px") if canonical_row else None

    if AUTO_DETECT_MODES_FROM_DISK:
        modes_this = None
    else:
        modes_this = (canonical_row.get("token_edit_modes") if canonical_row else None) or []

    # determine number of columns (modes)
    if AUTO_DETECT_MODES_FROM_DISK:
        # max modes across experiments for this image_id (keeps grid stable)
        max_modes = 0
        for name in exp_names:
            images_root = Path(EXPERIMENTS[name].images_dir)
            ps_tmp = paths_for_sample(images_root, image_id, modes=None)
            max_modes = max(max_modes, len(ps_tmp["recon_edits"].keys()))
        ncols = max(1, max_modes)
    else:
        ncols = max(1, len(modes_this))

    rows_per_exp = 4  # header + base + edits + heatmaps
    nrows = len(exp_names) * rows_per_exp

    height_ratios = []
    for _ in exp_names:
        height_ratios += [HEADER_H, 1.0, 1.0, 1.0]

    fig = plt.figure(figsize=(COL_W * ncols, ROW_H * nrows))
    gs = gridspec.GridSpec(nrows=nrows, ncols=ncols, figure=fig, height_ratios=height_ratios)

    left_span, right_span = _span_split(ncols)

    for e_i, exp_name in enumerate(exp_names):
        exp = EXPERIMENTS[exp_name]
        images_root = Path(exp.images_dir)

        r0 = e_i * rows_per_exp

        # modes/paths for this exp & image
        if AUTO_DETECT_MODES_FROM_DISK:
            ps = paths_for_sample(images_root, image_id, modes=None)
            mode_keys = list(ps["recon_edits"].keys())
        else:
            ps = paths_for_sample(images_root, image_id, modes=modes_this)
            mode_keys = list(ps["recon_edits"].keys())  # expected same order as modes_this

        # load base images
        x_orig  = load_img_float01(ps["orig"]) if ps["orig"].exists() else None
        x_clean = load_img_float01(ps["recon_clean"]) if ps["recon_clean"].exists() else None

        # 1) HEADER (model name)
        ax_header = fig.add_subplot(gs[r0, :])
        ax_header.axis("off")
        ax_header.text(0.5, 0.5, exp_name, ha="center", va="center", fontsize=16, fontweight="bold")

        # 2) BASE ROW: original spans left half, recon spans right half
        ax_o = fig.add_subplot(gs[r0 + 1, :left_span])
        ax_c = fig.add_subplot(gs[r0 + 1, left_span:])
        _draw_cell(ax_o, "original", x_orig, bbox_px)
        _draw_cell(ax_c, "recon_clean", x_clean, bbox_px)

        # 3) EDITS ROW
        for c in range(ncols):
            ax = fig.add_subplot(gs[r0 + 2, c])
            if c >= len(mode_keys):
                ax.axis("off")
                continue
            m = mode_keys[c]
            p_edit = ps["recon_edits"].get(m)
            x_edit = load_img_float01(p_edit) if (p_edit is not None and p_edit.exists()) else None
            _draw_cell(ax, f"edit: {m}", x_edit, bbox_px)

        # 4) HEATMAPS ROW
        for c in range(ncols):
            ax = fig.add_subplot(gs[r0 + 3, c])
            if c >= len(mode_keys):
                ax.axis("off")
                continue
            m = mode_keys[c]
            p_edit = ps["recon_edits"].get(m)
            x_edit = load_img_float01(p_edit) if (p_edit is not None and p_edit.exists()) else None

            dmap = None if (x_clean is None or x_edit is None) else delta_map_l1_mean(x_clean, x_edit)
            _draw_heatmap_cell(ax, "Δ RMSE (per-pixel)", dmap, bbox_px, cmap=HEATMAP_CMAP)

    fig.suptitle(f"{image_id}  (patch_bbox_px)", fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    plt.show()
    plt.close(fig)  # important in notebooks so each image_id produces its own output


In [ ]:
print(SEED)

### Token-space sanity

#### Definitions

Let the token grid be
$$
z \in {0,\dots,K-1}^{H_{\text{tok}} \times W_{\text{tok}}}.
$$

Here $H_{\text{tok}}, W_{\text{tok}}$ denote the latent token grid resolution (VQGAN: $16 \times 16$ in our setup).

Let:

* `indices_clean` be the flattened clean token grid
* `indices_edit_by_mode[m]` be the flattened edited token grid for mode $m$
* $B_{\text{tok}} = (j_0,i_0,j_1,i_1)$ be the token-space bounding box that is edited
* $A_{\text{tok}} = (j_1-j_0)(i_1-i_0)$ be the token-space patch area

Define the token-patch mask
$$
M_{\text{tok}}(i,j) = \mathbf{1}[, i_0 \le i < i_1,; j_0 \le j < j_1 ,].
$$

For a given edit mode $m$, define the indicator of changed tokens (in flattened indexing) as
$$
\Delta^{(m)}(k) = \mathbf{1}!\left[\text{indices}^{(m)}*{\text{edit}}(k) \neq \text{indices}*{\text{clean}}(k)\right].
$$

We track three edit fractions per mode:

* Overall:
  $$
  f_{\text{all}}^{(m)}=\frac{1}{H_{\text{tok}}W_{\text{tok}}}\sum_k \Delta^{(m)}(k).
  $$

* Inside the patch:
  $$
  f_{\text{in}}^{(m)}=\frac{1}{A_{\text{tok}}}\sum_{(i,j)\in B_{\text{tok}}}\Delta^{(m)}(i,j).
  $$

* Outside the patch:
  $$
  f_{\text{out}}^{(m)}=\frac{1}{H_{\text{tok}}W_{\text{tok}}-A_{\text{tok}}}\sum_{(i,j)\notin B_{\text{tok}}}\Delta^{(m)}(i,j).
  $$


In [ ]:
# ---- token-space sanity: parse indices per experiment + run ----
def _exp_get(exp, key, default=None):
    """Get attribute from dict-like or object-like experiment."""
    if isinstance(exp, dict):
        return exp.get(key, default)
    return getattr(exp, key, default)

def _infer_tok_hw(rec):
    hw = rec.get("token_grid_hw", None)
    if hw is None:
        return None
    return int(hw[0]), int(hw[1])

def _make_patch_mask_flat(Htok, Wtok, bb_tok):
    # bb_tok = (j0, i0, j1, i1)
    j0, i0, j1, i1 = map(int, bb_tok)
    mask = np.zeros((Htok, Wtok), dtype=bool)
    mask[i0:i1, j0:j1] = True
    return mask.reshape(-1)

def token_sanity_df(records):
    rows = []
    for r in records:
        z_clean = r.get("indices_clean", None)
        edits_by_mode = r.get("indices_edit_by_mode", None)
        modes = r.get("token_edit_modes", None)
        bb_tok = r.get("patch_bbox_tok", None)
        hw = _infer_tok_hw(r)

        if z_clean is None or edits_by_mode is None or modes is None:
            continue
        if bb_tok is None or hw is None:
            continue

        Htok, Wtok = hw
        total_area = int(Htok * Wtok)
        patch_mask = _make_patch_mask_flat(Htok, Wtok, bb_tok)
        patch_area = int(patch_mask.sum())
        patch_frac = patch_area / float(total_area)

        zc = np.asarray(z_clean, dtype=np.int64)

        for m in modes:
            ze_list = edits_by_mode.get(m, None)
            if ze_list is None:
                continue
            ze = np.asarray(ze_list, dtype=np.int64)
            changed = (ze != zc)

            overall = float(changed.mean())
            inside  = float(changed[patch_mask].mean()) if patch_area > 0 else np.nan
            outside = float(changed[~patch_mask].mean()) if patch_area < total_area else np.nan

            recon_overall_from_parts = inside * patch_frac + outside * (1.0 - patch_frac)

            rows.append({
                "image_id": r.get("image_id", None),
                "mode": m,
                "Htok": Htok, "Wtok": Wtok,
                "patch_area_tok": patch_area,
                "patch_fraction": patch_frac,
                "edit_frac_overall": overall,
                "edit_frac_inside": inside,
                "edit_frac_outside": outside,
                "overall_minus_patchfrac": overall - patch_frac,
                "overall_minus_recon": overall - recon_overall_from_parts,
            })

    return pd.DataFrame(rows)

def _iter_experiments(experiments):
    """
    Accepts dict {name: exp} or list [exp,...].
    Prefers the global EXPERIMENTS if present.
    """
    if experiments is None:
        experiments = globals().get("EXPERIMENTS", None) or globals().get("experiments", None)

    if isinstance(experiments, dict):
        for name, exp in experiments.items():
            yield str(name), exp
        return

    for i, exp in enumerate(experiments or []):
        name = _exp_get(exp, "name", None) or _exp_get(exp, "experiment_name", None) or f"exp_{i:02d}"
        yield str(name), exp

def ensure_records_with_indices(exp, *, max_records=2000):
    """
    Builds a small sample WITH indices (indices_clean + indices_edit_by_mode).
    Stores it on the experiment as exp.records_tok and returns it.
    """
    # already present?
    recs = _exp_get(exp, "records_tok", None)
    if recs is not None and len(recs) > 0:
        return recs

    meta_files = _exp_get(exp, "meta_files", None)
    if meta_files is None:
        return None

    recs = list(iter_meta_records(meta_files, keep_indices=True, max_records=max_records))

    # attach back for reuse
    if isinstance(exp, dict):
        exp["records_tok"] = recs
    else:
        setattr(exp, "records_tok", recs)

    return recs

def run_token_sanity(experiments=None, *, max_records=2000, topk=10):
    df_tok_by_exp = {}

    for name, exp in _iter_experiments(experiments):
        records = ensure_records_with_indices(exp, max_records=max_records)

        if not records:
            print(f"[{name}] could not build records_tok (missing meta_files or empty); skipping")
            continue

        # schema guard
        sample = next((r for r in records if isinstance(r, dict)), None)
        need = {"indices_clean", "indices_edit_by_mode", "token_edit_modes", "patch_bbox_tok", "token_grid_hw"}
        if sample is None or any(k not in sample for k in need):
            print(f"[{name}] records_tok exists, required fields missing; skipping")
            continue

        df_tok = token_sanity_df(records)
        df_tok_by_exp[name] = df_tok

        print(f"\n[{name}] Token sanity rows (image,mode): {len(df_tok)}  (parsed {len(records)} records)")
        display(
            df_tok.groupby("mode")[[
                "patch_fraction",
                "edit_frac_overall",
                "edit_frac_inside",
                "edit_frac_outside",
                "overall_minus_patchfrac",
            ]].describe(percentiles=[0.5, 0.9, 0.99])
        )

        print("\nLargest edit_frac_outside:")
        display(
            df_tok.sort_values("edit_frac_outside", ascending=False)
                  .head(topk)[["image_id","mode","patch_fraction","edit_frac_overall","edit_frac_inside","edit_frac_outside"]]
        )

    return df_tok_by_exp

# vqgan first, then llamagen (dict order)
df_tok_by_exp = run_token_sanity(EXPERIMENTS if "EXPERIMENTS" in globals() else experiments,
                                 max_records=2000, topk=10)


## Pixel-space locality metrics

We evaluate decoder locality **per token-edit mode** by comparing the clean reconstruction to each edited reconstruction.

#### Definition of the change map

Let:

- $x_{\text{clean}}$ be the decoded image from `1_recon_clean.png`
- $x_{\text{edit}}^{(m)}$ be the decoded image from `2_recon_token_edit_<mode>.png` for mode $m \in \texttt{token\_edit\_modes}$

Define the per-pixel change magnitude (channel-averaged $L^1$):

$$
\Delta^{(m)}(u,v) = \frac{1}{3}\sum_{c=1}^{3}\left|x_{\text{edit}}^{(m)}(u,v,c) - x_{\text{clean}}(u,v,c)\right|.
$$

#### Inside vs outside change

Let `patch_bbox_px = (x_0,y_0,x_1,y_1)` and define:

- inside region: $R = [x_0,x_1)\times[y_0,y_1)$
- outside region: $R^c$ (all other pixels)

We compute:

$$
\text{inside\_change}^{(m)} = \mathbb{E}\!\left[\Delta^{(m)} \mid (u,v)\in R\right],
\qquad
\text{outside\_change}^{(m)} = \mathbb{E}\!\left[\Delta^{(m)} \mid (u,v)\in R^c\right].
$$

#### Leakage ratio

$$
\text{leakage\_ratio}^{(m)} =
\frac{\text{outside\_change}^{(m)}}{\text{inside\_change}^{(m)}+\varepsilon}.
$$

A small leakage ratio indicates decoder locality: the edit mainly affects pixels inside the target region.

#### Why we also track tails

Even if the mean outside change is small, there can be rare global ripples.

So we compute high percentiles of $\Delta^{(m)}$ **outside** the patch (`outside_p99`, `outside_p999`) to detect occasional but strong leakage.

#### Δ heatmaps

For each mode $m$, the map $\Delta^{(m)}$ is a nonnegative “change magnitude map”: bright means “changed a lot”, dark means “unchanged”.

Heatmaps provide:

1) **Immediate locality sanity**  
If the decoder were perfectly local, the bright region would mostly sit inside the box. Brightness outside the box is leakage.

2) **Leakage type, not just leakage amount**  
The mean outside change can be small while still having a few bright streaks far away. A heatmap can reveal:
- boundary-only spill (a halo around the patch)
- global ripples (low-amplitude change everywhere)
- structure-coupled leakage (changes that follow object contours/textures)

3) **Detect mapping bugs**  
If the bright region is consistently shifted relative to the box, the `patch_bbox_px` mapping is wrong. If it aligns but still spills, that reflects real decoder coupling.

In [ ]:
# core pixel locality metrics per (experiment, image_id, mode) — parallel CPU + tqdm

EPS = 1e-8
MAX_IMAGES_FOR_PIXEL_METRICS = None          # e.g. 5000 for speed, or None for all
AUTO_DETECT_MODES_FROM_DISK = False         # True -> ignore metadata modes, detect from filenames on disk

def inside_outside_stats(d: np.ndarray, bbox_px):
    H, W = d.shape
    x0, y0, x1, y1 = map(int, bbox_px)

    x0 = max(0, min(W, x0)); x1 = max(0, min(W, x1))
    y0 = max(0, min(H, y0)); y1 = max(0, min(H, y1))

    inside = d[y0:y1, x0:x1]
    inside_sum = float(inside.sum())
    inside_cnt = int(inside.size)

    total_sum = float(d.sum())
    total_cnt = int(d.size)

    outside_sum = total_sum - inside_sum
    outside_cnt = total_cnt - inside_cnt

    inside_mean = inside_sum / (inside_cnt + EPS)
    outside_mean = outside_sum / (outside_cnt + EPS)
    leakage_ratio = outside_mean / (inside_mean + EPS)

    outside_vals = []
    if y0 > 0: outside_vals.append(d[:y0, :].ravel())
    if y1 < H: outside_vals.append(d[y1:, :].ravel())
    if x0 > 0: outside_vals.append(d[y0:y1, :x0].ravel())
    if x1 < W: outside_vals.append(d[y0:y1, x1:].ravel())

    outside_flat = np.concatenate(outside_vals) if outside_vals else np.array([], dtype=d.dtype)
    outside_p99  = float(np.percentile(outside_flat, 99))   if outside_flat.size else 0.0
    outside_p999 = float(np.percentile(outside_flat, 99.9)) if outside_flat.size else 0.0

    return inside_mean, outside_mean, leakage_ratio, outside_p99, outside_p999

def _delta_map(x_clean, x_edit):
    # channel-averaged L1
    return np.mean(np.abs(x_edit - x_clean), axis=-1)

def _process_one_image(image_id, bbox_px, modes, images_root_str):
    """
    Worker: compute metrics for all modes for one image.
    Returns list[dict] rows.
    """
    from pathlib import Path
    import numpy as np

    images_root = Path(images_root_str)

    ps = paths_for_sample(images_root, image_id, modes=modes)
    if not ps["recon_clean"].exists():
        return []

    x_clean = load_img_float01(ps["recon_clean"])
    out_rows = []

    for m, p_edit in ps["recon_edits"].items():
        if not p_edit.exists():
            continue

        x_edit = load_img_float01(p_edit)
        d = _delta_map(x_clean, x_edit)

        inside_mean, outside_mean, leak, out_p99, out_p999 = inside_outside_stats(d, bbox_px)
        out_rows.append({
            "image_id": image_id,
            "mode": m,
            "inside_change": inside_mean,
            "outside_change": outside_mean,
            "leakage_ratio": leak,
            "outside_p99": out_p99,
            "outside_p999": out_p999,
        })

    return out_rows

def _get_records_for_exp(exp_name: str, exp):
    # Prefer prebuilt index per experiment
    if "records_index" in globals() and isinstance(records_index, dict) and exp_name in records_index:
        return records_index[exp_name]

    # Fallback: build from metadata if available
    if hasattr(exp, "meta_files"):
        max_records = globals().get("MAX_RECORDS", None)
        return build_index(exp.meta_files, max_records=max_records)

    # Last resort: try a global records_small
    if "records_small" in globals():
        return records_small

    raise RuntimeError(f"No records available for exp={exp_name}. Expected records_index[...] or exp.meta_files or records_small.")

def compute_pixel_metrics_for_experiment(
    exp_name: str,
    exp,
    *,
    max_images: int | None = MAX_IMAGES_FOR_PIXEL_METRICS,
    auto_detect_modes: bool = AUTO_DETECT_MODES_FROM_DISK,
):
    images_root = Path(exp.images_dir) if hasattr(exp, "images_dir") else Path(exp)

    recs = _get_records_for_exp(exp_name, exp)

    # Build tasks
    tasks = []
    for r in recs:
        image_id = r.get("image_id")
        bbox_px  = r.get("patch_bbox_px")

        if auto_detect_modes:
            modes = None
        else:
            modes = r.get("token_edit_modes")

        if image_id is None or bbox_px is None:
            continue
        if (modes is not None) and (not modes):
            continue

        tasks.append((image_id, bbox_px, modes))

    if max_images is not None:
        tasks = tasks[:max_images]

    if not tasks:
        return pd.DataFrame()

    workers = min(os.cpu_count() or 4, len(tasks))
    rows = []

    with ProcessPoolExecutor(max_workers=workers) as ex:
        futs = [
            ex.submit(_process_one_image, img_id, bbox_px, modes, str(images_root))
            for (img_id, bbox_px, modes) in tasks
        ]

        for fut in tqdm(as_completed(futs), total=len(futs), desc=f"[{exp_name}] pixel metrics"):
            rows.extend(fut.result())

    df = pd.DataFrame(rows)
    if len(df):
        df.insert(0, "experiment", exp_name)
    return df

# ---- run for all experiments ----
df_pix_by_exp = {}
dfs = []

for exp_name, exp in EXPERIMENTS.items():
    print(f"\n[{exp_name}] computing pixel metrics | images_root={getattr(exp, 'images_dir', exp)}")
    df_e = compute_pixel_metrics_for_experiment(exp_name, exp)
    df_pix_by_exp[exp_name] = df_e
    dfs.append(df_e)

df_pix_all = pd.concat([d for d in dfs if len(d)], ignore_index=True) if any(len(d) for d in dfs) else pd.DataFrame()

print("\nDone.")
print("Total experiments:", len(EXPERIMENTS))
print("Total pixel-metric rows:", len(df_pix_all))
display(df_pix_all.head(3))

# Per-experiment summaries
if len(df_pix_all):
    display(
        df_pix_all.groupby(["experiment", "mode"])[
            ["inside_change", "outside_change", "leakage_ratio", "outside_p99", "outside_p999"]
        ].describe(percentiles=[0.5, 0.9, 0.99])
    )

In [ ]:
CACHE_DIR = Path("/exp/analysis_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def _exp_dir_basename(exp):
    # exp.images_dir is .../<exp_dir>/images
    p = Path(getattr(exp, "images_dir", exp))
    if p.name == "images":
        return p.parent.name
    return p.name

# ---- SAVE (one file per experiment) ----
for exp_name, exp in EXPERIMENTS.items():
    if "df_pix_by_exp" not in globals() or exp_name not in df_pix_by_exp:
        print(f"[skip] {exp_name}: df_pix_by_exp missing")
        continue

    df = df_pix_by_exp[exp_name]
    fname = f"{_exp_dir_basename(exp)}__pixel_metrics.csv.gz"
    out = CACHE_DIR / fname
    df.to_csv(out, index=False, compression="gzip")
    print(f"[saved] {exp_name}: {len(df):,} rows -> {out}")

In [ ]:
# ---- LOAD (rebuild df_pix_by_exp + df_pix_all from cache) ----
df_pix_by_exp = {}
for exp_name, exp in EXPERIMENTS.items():
    fname = f"{_exp_dir_basename(exp)}__pixel_metrics.csv.gz"
    p = CACHE_DIR / fname
    if not p.exists():
        print(f"[missing] {exp_name}: {p}")
        continue
    df_pix_by_exp[exp_name] = pd.read_csv(p)
    print(f"[loaded] {exp_name}: {len(df_pix_by_exp[exp_name]):,} rows <- {p}")

df_pix_all = pd.concat(df_pix_by_exp.values(), ignore_index=True) if df_pix_by_exp else pd.DataFrame()
print("df_pix_all rows:", len(df_pix_all))

In [ ]:
INSIDE_MIN = 0.02
K_TABLE = 10

df = df_pix_all.copy()
df["leakage_ratio_safe"] = df["outside_change"] / (df["inside_change"] + 1e-8)
df_safe = df[df["inside_change"] >= INSIDE_MIN].copy()

In [ ]:
summary = (
    df_safe
    .groupby(["experiment", "mode"])
    .agg(
        n=("image_id", "count"),
        inside_mean=("inside_change", "mean"),
        outside_mean=("outside_change", "mean"),
        lr_med=("leakage_ratio_safe", "median"),
        lr_p90=("leakage_ratio_safe", lambda x: x.quantile(0.90)),
        lr_p99=("leakage_ratio_safe", lambda x: x.quantile(0.99)),
        out_p99_med=("outside_p99", "median"),
        out_p999_med=("outside_p999", "median"),
    )
    .sort_values(["experiment", "lr_med"])
)

summary


leakage_ratio is the raw, physical definition:
outside_change ÷ inside_change.
this is the true mathematical ratio and is correct everywhere.

leakage_ratio_safe is the analysis-facing metric:
outside_change ÷ (inside_change + ε), evaluated only when inside_change ≥ INSIDE_MIN.
It stabilizes ranking, summaries, and plots by preventing tiny denominators from dominating results.

In [ ]:
# ranked tables: best/worst leakage_ratio_safe per (experiment, mode)

cols = [
    "experiment", "image_id", "mode",
    "inside_change", "outside_change",
    "leakage_ratio", "leakage_ratio_safe",
    "outside_p99", "outside_p999",
]

best = (
    df_safe.sort_values("leakage_ratio_safe")
    .groupby(["experiment", "mode"])
    .head(K_TABLE)[cols]
)

worst = (
    df_safe.sort_values("leakage_ratio_safe", ascending=False)
    .groupby(["experiment", "mode"])
    .head(K_TABLE)[cols]
)

best, worst


In [ ]:
# ranked tables: worst absolute leakage (outside_change) per (experiment, mode)

cols = [
    "experiment", "image_id", "mode",
    "inside_change", "outside_change",
    "leakage_ratio", "leakage_ratio_safe",
    "outside_p99", "outside_p999",
]

worst_abs = (
    df.sort_values("outside_change", ascending=False)
    .groupby(["experiment", "mode"])
    .head(K_TABLE)[cols]
)

worst_abs


In [ ]:
def paths_for_sample(images_root: Path, image_id: str, mode: Optional[str] = None):
    d = images_root / image_id

    orig = d / "0_original.png"
    clean = d / "1_recon_clean.png"

    if mode is None:
        # legacy fallback
        edit = d / "2_recon_token_edit.png"
        if not edit.exists():
            # or pick the first mode on disk if legacy is absent
            cands = sorted(d.glob("2_recon_token_edit_*.png"))
            edit = cands[0] if cands else edit
    else:
        edit = d / f"2_recon_token_edit_{mode}.png"

    return {"orig": orig, "recon_clean": clean, "recon_edit": edit}


In [ ]:
def show_triplet(image_id, bbox_px, images_root: Path, mode: Optional[str] = None, title_prefix: str = ""):
    ps = paths_for_sample(images_root, image_id, mode=mode)

    x_orig = load_img_float01(ps["orig"]) if ps["orig"].exists() else None
    x_clean = load_img_float01(ps["recon_clean"])
    x_edit  = load_img_float01(ps["recon_edit"])

    d = delta_map(x_clean, x_edit)

    cols = []
    if x_orig is not None:
        cols.append(("orig", x_orig))
    cols += [("recon_clean", x_clean), (f"edit:{mode}" if mode else "recon_token_edit", x_edit)]

    ncols = len(cols) + 1
    fig, axes = plt.subplots(1, ncols, figsize=(4.5 * ncols, 4))
    if ncols == 1:
        axes = [axes]

    for i, (title, img) in enumerate(cols):
        ax = axes[i]
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title)
        if bbox_px is not None and title != "orig":
            x0, y0, x1, y1 = map(int, bbox_px)
            rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, fill=False)
            ax.add_patch(rect)

    ax = axes[-1]
    ax.imshow(d, cmap="magma")
    ax.axis("off")
    ax.set_title("Δ heatmap")
    if bbox_px is not None:
        x0, y0, x1, y1 = map(int, bbox_px)
        rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, fill=False, edgecolor="cyan")
        ax.add_patch(rect)

    plt.suptitle(f"{title_prefix}{image_id}")
    plt.tight_layout()
    plt.show()


In [ ]:
K = 2  # how many best/worst per (experiment, mode)

# build {experiment_name: images_root} using your existing helpers
exp_items = [(name, _resolve_images_root(exp)) for name, exp in _iter_experiments(experiments)]
images_root_by_experiment = {name: root for name, root in exp_items}

df_rank = df_safe.copy()

for (exp, mode), sub in df_rank.groupby(["experiment", "mode"]):
    best_ids  = sub.sort_values("leakage_ratio_safe").head(K)["image_id"].tolist()
    worst_ids = sub.sort_values("leakage_ratio_safe", ascending=False).head(K)["image_id"].tolist()

    images_root = images_root_by_experiment[exp]

    print(f"\n=== {exp} | mode: {mode} ===")
    print("Best (lowest leakage_ratio_safe):")
    for iid in best_ids:
        show_triplet(iid, bbox_map.get(iid), images_root, mode=mode, title_prefix=f"{exp} | {mode} | ")

    print("Worst (highest leakage_ratio_safe):")
    for iid in worst_ids:
        show_triplet(iid, bbox_map.get(iid), images_root, mode=mode, title_prefix=f"{exp} | {mode} | ")


## Reconstruction quality metrics (PSNR / SSIM / LPIPS)

To complement locality metrics, we measure reconstruction fidelity between the clean decode and each edited decode for every token-edit mode.

For each sample and mode `m`, let:

- `x_clean`: clean reconstruction  
- `x_edit^(m)`: reconstruction after token patch edit

We evaluate on two spatial supports:

- `Ω`: full image domain  
- `Ω_patch ⊂ Ω`: patch domain defined by `patch_bbox_px`

### Metrics

- **PSNR** (↑ higher is better)  
- **SSIM** (↑ higher is better)  
- **LPIPS** (↓ lower is better)

Thus, for each sample and mode `m`, we compute:

- `psnr_full`, `ssim_full`, `lpips_full`
- `psnr_patch`, `ssim_patch`, `lpips_patch`

### Interpretation

- Full-image metrics reflect global perceptual distortion.
- Patch metrics isolate the local effect of the intervention within the edited region.
- The gap between full and patch metrics indicates how concentrated the degradation is inside the edited patch.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

def _clamp_bbox(bbox_px, H, W):
    x0, y0, x1, y1 = map(int, bbox_px)
    x0 = max(0, min(W, x0)); x1 = max(0, min(W, x1))
    y0 = max(0, min(H, y0)); y1 = max(0, min(H, y1))
    if x1 <= x0 or y1 <= y0:
        return None
    return x0, y0, x1, y1

def build_iq_tasks(experiments, records_index, max_images_per_exp=1000, mode_subset=None):
    mode_subset = set(mode_subset) if mode_subset is not None else None
    tasks = []

    for exp_name, exp in experiments.items():
        images_root = Path(exp.images_dir)
        recs = records_index[exp_name]
        if max_images_per_exp is not None:
            recs = recs[:max_images_per_exp]

        for r in recs:
            image_id = r.get("image_id")
            bbox_px = r.get("patch_bbox_px")
            modes = list(r.get("token_edit_modes") or [])
            if image_id is None or bbox_px is None or len(modes) == 0:
                continue

            if mode_subset is not None:
                modes = [m for m in modes if m in mode_subset]
                if len(modes) == 0:
                    continue

            tasks.append({
                "experiment": exp_name,
                "images_root": str(images_root),
                "image_id": image_id,
                "bbox_px": tuple(map(int, bbox_px)),
                "modes": tuple(modes),
            })

    return tasks


In [ ]:
#  multiprocess only for PSNR/SSIM (stable)
from concurrent.futures import ProcessPoolExecutor
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def _psnr_ssim_only_task(task):
    rows = []
    exp_name = task["experiment"]
    images_root = Path(task["images_root"])
    image_id = task["image_id"]
    bbox_px = task["bbox_px"]
    modes = task["modes"]

    d = images_root / image_id
    p_clean = d / "1_recon_clean.png"
    if not p_clean.exists():
        return rows

    x_clean = load_img_float01(p_clean)
    H, W = x_clean.shape[:2]
    bb = _clamp_bbox(bbox_px, H, W)
    if bb is None:
        return rows
    x0, y0, x1, y1 = bb
    clean_patch = x_clean[y0:y1, x0:x1, :]

    def _safe_ssim(a, b):
        h, w = a.shape[:2]
        win = min(7, h, w)
        if win % 2 == 0:
            win -= 1
        if win < 3:
            return np.nan
        return float(structural_similarity(a, b, channel_axis=2, data_range=1.0, win_size=win))

    for mode in modes:
        p_edit = d / f"2_recon_token_edit_{mode}.png"
        if not p_edit.exists():
            continue
        x_edit = load_img_float01(p_edit)
        if x_edit.shape != x_clean.shape:
            continue

        edit_patch = x_edit[y0:y1, x0:x1, :]

        rows.append({
            "experiment": exp_name,
            "image_id": image_id,
            "mode": mode,
            "psnr_full": float(peak_signal_noise_ratio(x_clean, x_edit, data_range=1.0)),
            "ssim_full": _safe_ssim(x_clean, x_edit),
            "psnr_patch": float(peak_signal_noise_ratio(clean_patch, edit_patch, data_range=1.0)),
            "ssim_patch": _safe_ssim(clean_patch, edit_patch),
            "clean_path": str(p_clean),
            "edit_path": str(p_edit),
            "x0": x0, "y0": y0, "x1": x1, "y1": y1,
        })
    return rows

def run_psnr_ssim_mp(tasks, num_workers=None, chunksize=16):
    if num_workers is None:
        num_workers = min(os.cpu_count() or 4, len(tasks))
    rows = []
    with ProcessPoolExecutor(max_workers=max(1, num_workers)) as ex:
        for out in tqdm(ex.map(_psnr_ssim_only_task, tasks, chunksize=chunksize), total=len(tasks), desc="PSNR/SSIM MP"):
            rows.extend(out)
    return pd.DataFrame(rows)


In [ ]:
#  run PSNR/SSIM multiprocess
MAX_IMAGES_PER_EXP = 50000
N_WORKERS = min(14, os.cpu_count() or 4)

tasks_iq = build_iq_tasks(EXPERIMENTS, records_index, max_images_per_exp=MAX_IMAGES_PER_EXP, mode_subset=None)
print("tasks:", len(tasks_iq))

df_iq = run_psnr_ssim_mp(tasks_iq, num_workers=N_WORKERS, chunksize=16)
print("rows after PSNR/SSIM:", len(df_iq))
display(df_iq.head(3))


In [ ]:
from pathlib import Path
import gzip

CACHE_DIR = Path("/exp/analysis_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OUT_FP = CACHE_DIR / "decoder_locality_psnr_ssim_all.csv.gz"

df_iq.to_csv(OUT_FP, index=False, compression="gzip")

print(f"[saved] {len(df_iq):,} rows -> {OUT_FP}")

In [ ]:
N_PER_GROUP = 5000  # example

df_lp = (
    df_iq.groupby(["experiment", "mode"], group_keys=False)
    .apply(lambda g: g.sample(n=min(N_PER_GROUP, len(g)), random_state=0))
    .reset_index(drop=True)
)

print("df_lp rows:", len(df_lp))
display(df_lp.head(3))

In [ ]:
# LPIPS grouped-by-image on df_lp, not df_iq
import lpips
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_IMAGES = 16
USE_AMP = torch.cuda.is_available()

lpips_fn = lpips.LPIPS(net="alex").to(DEVICE).eval()

def _to_lpips_tensor_batch(imgs01):
    arr = np.stack([img.transpose(2, 0, 1) for img in imgs01], axis=0)
    t = torch.from_numpy(arr).float().to(DEVICE, non_blocking=True)
    return t * 2.0 - 1.0

df_lp = df_lp.reset_index(drop=False).rename(columns={"index": "row_id"})

group_cols = ["experiment", "image_id", "clean_path", "x0", "y0", "x1", "y1"]
groups = list(df_lp.groupby(group_cols, sort=False))

lpips_full = np.empty(len(df_lp), dtype=np.float32)
lpips_patch = np.empty(len(df_lp), dtype=np.float32)

for g0 in tqdm(range(0, len(groups), BATCH_IMAGES), desc="LPIPS grouped"):
    batch_groups = groups[g0:g0 + BATCH_IMAGES]

    full_clean_imgs = []
    full_edit_imgs = []
    patch_clean_imgs = []
    patch_edit_imgs = []
    row_ids = []

    for _, gdf in batch_groups:
        clean_path = Path(gdf["clean_path"].iloc[0])
        x_clean = load_img_float01(clean_path)

        x0 = int(gdf["x0"].iloc[0])
        y0 = int(gdf["y0"].iloc[0])
        x1 = int(gdf["x1"].iloc[0])
        y1 = int(gdf["y1"].iloc[0])

        clean_patch = x_clean[y0:y1, x0:x1, :]

        for r in gdf.itertuples(index=False):
            x_edit = load_img_float01(Path(r.edit_path))

            full_clean_imgs.append(x_clean)
            full_edit_imgs.append(x_edit)
            patch_clean_imgs.append(clean_patch)
            patch_edit_imgs.append(x_edit[y0:y1, x0:x1, :])
            row_ids.append(r.row_id)

    with torch.no_grad():
        t_clean_full = _to_lpips_tensor_batch(full_clean_imgs)
        t_edit_full = _to_lpips_tensor_batch(full_edit_imgs)
        t_clean_patch = _to_lpips_tensor_batch(patch_clean_imgs)
        t_edit_patch = _to_lpips_tensor_batch(patch_edit_imgs)

        if USE_AMP:
            with torch.cuda.amp.autocast():
                lp_f = lpips_fn(t_clean_full, t_edit_full).view(-1)
                lp_p = lpips_fn(t_clean_patch, t_edit_patch).view(-1)
        else:
            lp_f = lpips_fn(t_clean_full, t_edit_full).view(-1)
            lp_p = lpips_fn(t_clean_patch, t_edit_patch).view(-1)

        lp_f = lp_f.detach().cpu().numpy()
        lp_p = lp_p.detach().cpu().numpy()

    lpips_full[row_ids] = lp_f
    lpips_patch[row_ids] = lp_p

df_lp["lpips_full"] = lpips_full
df_lp["lpips_patch"] = lpips_patch

print("done. rows:", len(df_lp))
display(df_lp.head(3))

In [ ]:
from pathlib import Path

CACHE_DIR = Path("/exp/analysis_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OUT_FP_LP = CACHE_DIR / "decoder_locality_lpips_sample.csv.gz"

df_lp.to_csv(OUT_FP_LP, index=False, compression="gzip")

print(f"[saved] {len(df_lp):,} rows -> {OUT_FP_LP}")

In [ ]:
from pathlib import Path
import pandas as pd

CACHE_DIR = Path("/exp/analysis_cache")

FP_IQ = CACHE_DIR / "decoder_locality_psnr_ssim_all.csv.gz"
FP_LP = CACHE_DIR / "decoder_locality_lpips_sample.csv.gz"

df_iq = pd.read_csv(FP_IQ)
df_lp = pd.read_csv(FP_LP)

print(f"[loaded] df_iq: {len(df_iq):,} rows <- {FP_IQ}")
print(f"[loaded] df_lp: {len(df_lp):,} rows <- {FP_LP}")

# keep only one LPIPS row per key, just in case the notebook state produced duplicates
df_lp_merge = (
    df_lp[["experiment", "image_id", "mode", "lpips_full", "lpips_patch"]]
    .drop_duplicates(subset=["experiment", "image_id", "mode"])
    .copy()
)

df_iq_lp = df_iq.merge(
    df_lp_merge,
    on=["experiment", "image_id", "mode"],
    how="left",
)

print("df_iq_lp rows:", len(df_iq_lp))
display(df_iq_lp.head(3))

In [ ]:
summary_iq = (
    df_iq_lp.groupby(["experiment", "mode"])
    .agg(
        n=("image_id", "count"),
        n_lpips=("lpips_patch", lambda s: s.notna().sum()),
        psnr_full_mean=("psnr_full", "mean"),
        psnr_patch_mean=("psnr_patch", "mean"),
        ssim_full_mean=("ssim_full", "mean"),
        ssim_patch_mean=("ssim_patch", "mean"),
        lpips_full_mean=("lpips_full", "mean"),
        lpips_patch_mean=("lpips_patch", "mean"),
    )
    .sort_values(["experiment", "lpips_patch_mean"])
)

display(summary_iq)

In [ ]:
OUT_FP_MERGED = CACHE_DIR / "decoder_locality_psnr_ssim_lpips_merged.csv.gz"
df_iq_lp.to_csv(OUT_FP_MERGED, index=False, compression="gzip")
print(f"[saved] {len(df_iq_lp):,} rows -> {OUT_FP_MERGED}")

In [ ]:
summary_iq = (
    df_iq_lp.groupby(["experiment", "mode"])
    .agg(
        n=("image_id", "count"),
        n_lpips=("lpips_patch", lambda s: s.notna().sum()),

        psnr_full_mean=("psnr_full", "mean"),
        psnr_full_std=("psnr_full", "std"),
        psnr_full_var=("psnr_full", "var"),
        psnr_full_q1=("psnr_full", lambda s: s.quantile(0.25)),
        psnr_full_q3=("psnr_full", lambda s: s.quantile(0.75)),

        psnr_patch_mean=("psnr_patch", "mean"),
        psnr_patch_std=("psnr_patch", "std"),
        psnr_patch_var=("psnr_patch", "var"),
        psnr_patch_q1=("psnr_patch", lambda s: s.quantile(0.25)),
        psnr_patch_q3=("psnr_patch", lambda s: s.quantile(0.75)),

        ssim_full_mean=("ssim_full", "mean"),
        ssim_full_std=("ssim_full", "std"),
        ssim_full_var=("ssim_full", "var"),
        ssim_full_q1=("ssim_full", lambda s: s.quantile(0.25)),
        ssim_full_q3=("ssim_full", lambda s: s.quantile(0.75)),

        ssim_patch_mean=("ssim_patch", "mean"),
        ssim_patch_std=("ssim_patch", "std"),
        ssim_patch_var=("ssim_patch", "var"),
        ssim_patch_q1=("ssim_patch", lambda s: s.quantile(0.25)),
        ssim_patch_q3=("ssim_patch", lambda s: s.quantile(0.75)),

        lpips_full_mean=("lpips_full", "mean"),
        lpips_full_std=("lpips_full", "std"),
        lpips_full_var=("lpips_full", "var"),
        lpips_full_q1=("lpips_full", lambda s: s.quantile(0.25)),
        lpips_full_q3=("lpips_full", lambda s: s.quantile(0.75)),

        lpips_patch_mean=("lpips_patch", "mean"),
        lpips_patch_std=("lpips_patch", "std"),
        lpips_patch_var=("lpips_patch", "var"),
        lpips_patch_q1=("lpips_patch", lambda s: s.quantile(0.25)),
        lpips_patch_q3=("lpips_patch", lambda s: s.quantile(0.75)),
    )
    .reset_index()
)

In [ ]:
import numpy as np

# PSNR / SSIM use full group count
for metric in ["psnr_full", "psnr_patch", "ssim_full", "ssim_patch"]:
    summary_iq[f"{metric}_sem"] = summary_iq[f"{metric}_std"] / np.sqrt(summary_iq["n"])

# LPIPS uses sampled count
for metric in ["lpips_full", "lpips_patch"]:
    summary_iq[f"{metric}_sem"] = summary_iq[f"{metric}_std"] / np.sqrt(summary_iq["n_lpips"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# use the aggregated table, not raw df_iq_lp
df_plot = summary_iq.copy()

rows = [
    ("psnr_full_mean",  "psnr_full_std",  "psnr"),
    ("psnr_patch_mean", "psnr_patch_std", "psnr"),
    ("ssim_full_mean",  "ssim_full_std",  "ssim"),
    ("ssim_patch_mean", "ssim_patch_std", "ssim"),
    ("lpips_full_mean", "lpips_full_std", "lpips"),
    ("lpips_patch_mean","lpips_patch_std","lpips"),
]

panel_grid = [
    [("psnr_full_mean",  "psnr_full_std"),  ("psnr_patch_mean",  "psnr_patch_std")],
    [("ssim_full_mean",  "ssim_full_std"),  ("ssim_patch_mean",  "ssim_patch_std")],
    [("lpips_full_mean", "lpips_full_std"), ("lpips_patch_mean", "lpips_patch_std")],
]

pretty_titles = {
    "psnr_full_mean": "PSNR full",
    "psnr_patch_mean": "PSNR patch",
    "ssim_full_mean": "SSIM full",
    "ssim_patch_mean": "SSIM patch",
    "lpips_full_mean": "LPIPS full",
    "lpips_patch_mean": "LPIPS patch",
}

metric_family = {
    "psnr_full_mean": "psnr",
    "psnr_patch_mean": "psnr",
    "ssim_full_mean": "ssim",
    "ssim_patch_mean": "ssim",
    "lpips_full_mean": "lpips",
    "lpips_patch_mean": "lpips",
}

colors = {
    "llamagen": "#5b74a8",
    "vqgan": "#c48f69",
}

baselines = {
    "vqgan": {
        "psnr": 19.65,
        "ssim": 0.486,
        "lpips": 0.286,
    },
    "llamagen": {
        "psnr": 20.79,
        "ssim": 0.5580,
        "lpips": 0.2281,
    },
}

mode_order = ["closest", "orthogonal", "random_uniform", "farthest"]
exp_order = ["llamagen", "vqgan"]

fig, axes = plt.subplots(3, 2, figsize=(14, 16))
fig.suptitle("Decoder-only edit fidelity by mode", fontsize=16, y=0.98)

bar_width = 0.35
x = np.arange(len(mode_order))

for i, row in enumerate(panel_grid):
    for j, (mean_col, err_col) in enumerate(row):
        ax = axes[i, j]
        fam = metric_family[mean_col]

        sub = df_plot[["experiment", "mode", mean_col, err_col]].copy()
        sub["mode"] = pd.Categorical(sub["mode"], categories=mode_order, ordered=True)
        sub = sub.sort_values(["mode", "experiment"])

        vals = {}
        errs = {}
        for exp in exp_order:
            tmp = (
                sub[sub["experiment"] == exp]
                .set_index("mode")
                .reindex(mode_order)
            )
            vals[exp] = tmp[mean_col].to_numpy()
            errs[exp] = tmp[err_col].to_numpy()

        ax.bar(
            x - bar_width/2,
            vals["llamagen"],
            width=bar_width,
            yerr=errs["llamagen"],
            capsize=4,
            color=colors["llamagen"],
            label="llamagen" if (i == 0 and j == 0) else None,
            alpha=0.95,
        )

        ax.bar(
            x + bar_width/2,
            vals["vqgan"],
            width=bar_width,
            yerr=errs["vqgan"],
            capsize=4,
            color=colors["vqgan"],
            label="vqgan" if (i == 0 and j == 0) else None,
            alpha=0.95,
        )

        # baseline horizontal lines
        ax.axhline(
            baselines["llamagen"][fam],
            color=colors["llamagen"],
            linestyle="--",
            linewidth=2,
            alpha=0.9,
            label="llamagen baseline" if (i == 0 and j == 0) else None,
        )
        ax.axhline(
            baselines["vqgan"][fam],
            color=colors["vqgan"],
            linestyle="--",
            linewidth=2,
            alpha=0.9,
            label="vqgan baseline" if (i == 0 and j == 0) else None,
        )

        row_max = np.nanmax([
            np.nanmax(vals["llamagen"] + errs["llamagen"]),
            np.nanmax(vals["vqgan"] + errs["vqgan"]),
            baselines["llamagen"][fam],
            baselines["vqgan"][fam],
        ])
        ax.set_ylim(0, row_max * 1.15)

        ax.set_title(pretty_titles[mean_col], fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(mode_order, rotation=30, ha="right")
        ax.set_xlabel("")
        ax.grid(axis="y", linestyle="--", alpha=0.5)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.02, 0.5), title="Legend")

plt.tight_layout()
plt.subplots_adjust(right=0.84)
plt.show()